# Analysis of model outputs


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import seaborn as sns
import os
import torch

In [ ]:
results_dir = "../../../results"
cv_strats = ["subject_loo", "recording_loo", "combined_loo"]

## First, baseline models

In [ ]:
baseline_date = "2025-12-30_20-31-38"
baseline_dir = os.path.join(results_dir, "BaselineCV/eSEEd_v2", baseline_date)
baseline_models = ["MeanEstimator","LightGBM", "GaussianNB", "MLP", "SVM"]


In [ ]:
# for each fold (left-out subject) in subject_loo, read y_pred.npy and y_true.npy
# aggregate all predictions and true values across folds, then plot histograms
for baseline_model in baseline_models:
    print("=========================================")
    print(f"\nBASELINE MODEL: {baseline_model}")

    for cv in cv_strats:
        print(f"\nCROSS-VALIDATION TYPE: {cv}")
        folds_dir = os.path.join(baseline_dir, cv, baseline_model)

        folds = sorted([f for f in os.listdir(folds_dir) if os.path.isdir(os.path.join(folds_dir, f))])
        
        # Initialize lists to collect all predictions and true values
        all_y_pred = []
        all_y_true = []
        
        for fold_idx, fold in enumerate(folds):
            fold_path = os.path.join(folds_dir, fold)
            pred_path = os.path.join(fold_path, "y_pred.npy")
            true_path = os.path.join(fold_path, "y_true.npy")
            
            if os.path.exists(pred_path) and os.path.exists(true_path):
                y_pred = np.load(pred_path)
                y_true = np.load(true_path)
                all_y_pred.append(y_pred)
                all_y_true.append(y_true)
            else:
                print(f"{pred_path} or {true_path} does not exist")
        
        # Concatenate all folds
        if all_y_pred and all_y_true:
            all_y_pred = np.concatenate(all_y_pred, axis=0)
            all_y_true = np.concatenate(all_y_true, axis=0)
            
            emotion_labels = ["anger", "disgust", "sadness", "tenderness"]
            emotion_colors_true = ["b", "g", "r", "c"]
            emotion_colors_pred = ["navy", "darkgreen", "darkred", "darkcyan"]
            
            # Plot all four emotions in a 2x2 grid
            fig, axes = plt.subplots(2, 2, figsize=(8, 4.5))
            for i, ax in enumerate(axes.ravel()):
                ax.hist(all_y_true[:, i], bins=30, alpha=0.5, label=f'True {emotion_labels[i]}',
                        density=True, color=emotion_colors_true[i], linestyle='-', edgecolor="grey")
                ax.hist(all_y_pred[:, i], bins=30, alpha=0.5, label=f'Pred {emotion_labels[i]}',
                        density=True, color=emotion_colors_pred[i], linestyle='--', edgecolor="black")
                ax.set_title(f'Emotion {emotion_labels[i]}')
                ax.set_xlabel('Intensity of emotion')
                ax.set_ylabel('Frequency')
                ax.legend()
            plt.suptitle(f'Baseline Model: {baseline_model} | CV Type: {cv}\nResults from all folds aggregated', y=1.02)
            plt.tight_layout()
            plt.show()
        else:
            print("No valid predictions found")

## GNN

In [ ]:
gnn_date = "RETAIN_2026-01-03_22-27-53"
gnn_dir = os.path.join(results_dir, "SpatioTemporalHeteroGNN/eSEEd_v2", gnn_date)
epoch_file_name = "epoch_010.pt"

In [ ]:
for cv in cv_strats:
    print(f"\nCROSS-VALIDATION TYPE: {cv}")
    folds_dir = os.path.join(gnn_dir, cv)
    if os.path.exists(folds_dir) is False:
        print(f"{folds_dir} does not exist")
        continue
    folds = sorted([f for f in os.listdir(folds_dir) if os.path.isdir(os.path.join(folds_dir, f))])

    y_pred_list = []
    y_true_list = []

    for fold_idx, fold in enumerate(folds):
        data_dir = os.path.join(folds_dir, fold, "data")
        epoch = torch.load(os.path.join(data_dir, epoch_file_name))
        y_pred = epoch['outputs'].cpu().numpy()
        y_true = epoch['targets'].cpu().numpy()
        y_pred_list.append(y_pred)
        y_true_list.append(y_true)
    
    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all =  np.concatenate(y_pred_list, axis=0)

    emotion_labels = ["sadness/anger?", "tenderness", "anger/sadness?", "disgust"]
    emotion_colors_true = ["r", "c", "b", "g"]
    emotion_colors_pred = ["darkred", "darkcyan", "navy", "darkgreen"]

    # Plot all four emotions in a 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(10, 6.4))
    for i, ax in enumerate(axes.ravel()):
        ax.hist(y_true_all[:, i], bins=30, alpha=0.7, label=f'True {emotion_labels[i]}',
                density=True, color=emotion_colors_true[i], linestyle='-', edgecolor="grey")
        ax.hist(y_pred_all[:, i], bins=30, alpha=0.5, label=f'Pred {emotion_labels[i]}',
                density=True, color=emotion_colors_pred[i], linestyle='--', edgecolor="black")
        ax.set_title(f'Emotion {emotion_labels[i]}')
        ax.legend()
    plt.suptitle(f'SpatioTemporalHeteroGNN Model | CV Type: {cv}\nResults from all folds aggregated', y=1.02)
    plt.tight_layout()
    plt.show()

## Difficult subjects based on MAE

In [85]:
for cv in cv_strats:
    print(f"\nCROSS-VALIDATION TYPE: {cv}")
    folds_dir = os.path.join(gnn_dir, cv)
    if os.path.exists(folds_dir) is False:
        print(f"{folds_dir} does not exist")
        continue
    folds = sorted([f for f in os.listdir(folds_dir) if os.path.isdir(os.path.join(folds_dir, f))])

    mae_dict = {}
    sd_dict = {}

    for fold_idx, fold in enumerate(folds):
        data_dir = os.path.join(folds_dir, fold, "data")
        epoch = torch.load(os.path.join(data_dir, epoch_file_name))

        metrics = epoch['metrics']
        metrics_agg = metrics['aggregated']
        metrics_per_emotion = metrics['per_emotion']

        for emotion in metrics_per_emotion.keys():
            mae = metrics_per_emotion[emotion]['mae']
            sd = metrics_per_emotion[emotion]['sd_error']
            if mae > 4:
                print(f"Fold {fold:<10} | Emotion: {emotion:<20} | MAE: {mae:>5.2f} | SD: {sd:>5.2f}")
            mae_dict.setdefault(emotion, []).append(mae)
            sd_dict.setdefault(emotion, []).append(sd)
        mae_dict.setdefault('aggregated', []).append(metrics_agg['mae'])
        sd_dict.setdefault('aggregated', []).append(metrics_agg['sd_error'])



CROSS-VALIDATION TYPE: subject_loo
Fold subject_1  | Emotion: emotion-anger        | MAE:  4.27 | SD:  4.18
Fold subject_1  | Emotion: emotion-sadness      | MAE:  4.69 | SD:  4.23
Fold subject_1  | Emotion: emotion-disgust      | MAE:  4.73 | SD:  4.82
Fold subject_12 | Emotion: emotion-anger        | MAE:  4.13 | SD:  4.47
Fold subject_12 | Emotion: emotion-disgust      | MAE:  4.26 | SD:  4.58
Fold subject_15 | Emotion: emotion-sadness      | MAE:  5.71 | SD:  0.80
Fold subject_16 | Emotion: emotion-anger        | MAE:  4.44 | SD:  4.57
Fold subject_16 | Emotion: emotion-sadness      | MAE:  4.54 | SD:  4.41
Fold subject_16 | Emotion: emotion-disgust      | MAE:  4.54 | SD:  4.36
Fold subject_17 | Emotion: emotion-sadness      | MAE:  4.36 | SD:  4.60
Fold subject_17 | Emotion: emotion-disgust      | MAE:  4.13 | SD:  4.74
Fold subject_18 | Emotion: emotion-sadness      | MAE:  4.09 | SD:  3.91
Fold subject_19 | Emotion: emotion-anger        | MAE:  4.03 | SD:  3.85
Fold subject_19